# Lot D — Silver : verdict avec le NLI affiné du Lot C (EXP-D-011)

**Auteur : Membre 5 (Hadil).** Même système qu'EXP-D-010 (recherche hybride, Aya Expanse 8B en 4 bits,
prompt v1, graine 42) ; **seul change le moteur NLI** : règles Bronze → modèle affiné du Lot C
(PR #21, EXP-C-011). Configurations et hypothèses fixées avant la mesure
(voir `rag_llm/silver_verdict_nli.py`) : rien n'est réglé ici.

Avant de commencer : *Exécution → Modifier le type d'exécution → **GPU T4***.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Jeton Hugging Face (Aya, accès contrôlé)
Même jeton *Read* qu'EXP-D-010, dans la case masquée, **jamais dans une cellule**.

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Colle ton jeton Hugging Face (lecture) : ").strip()
print("✅ jeton Hugging Face enregistré pour cette session")

## 2. Récupérer le projet (branche `lot-d-silver-nli`)
Jeton GitHub *classic* (case `repo`, 7 jours), dans la case masquée ; à supprimer sur GitHub après usage.

In [ ]:
import re, subprocess

token = getpass("Colle ton token GitHub puis appuie sur Entrée : ").strip()
url = f"https://{token}@github.com/SouissiOumaima/NewsCheckTN.git"
r = subprocess.run(["git", "clone", "-q", "-b", "lot-d-silver-nli", url], capture_output=True, text=True)
del token, url
print("✅ Projet téléchargé" if r.returncode == 0
      else "❌ " + re.sub(r"gh[pousr]_\w+", "ghp_***", r.stderr))
%cd NewsCheckTN
!git remote set-url origin https://github.com/SouissiOumaima/NewsCheckTN.git
!git log --oneline -1

In [ ]:
!pip install -q -r requirements-dev.txt sentence-transformers bitsandbytes accelerate -U transformers

## 3. Corpus de test (privé, reconstruit dans la session, jamais poussé sur GitHub)

In [ ]:
!python -m rag_llm.build_tunifact_eval
!python -m rag_llm.freeze_test --check

## 4. Checkpoint NLI affiné du Lot C (1,1 Go, depuis Google Drive)
Avant cette cellule : ouvre le lien Drive d'Achref, clic droit sur le dossier →
**Organiser → Ajouter un raccourci** → *Mon Drive*. La cellule cherche le dossier
`nli_mdeberta-ft_v3_lr1e-05_e6_seed42` et le copie dans `models/checkpoints/` (jamais sur GitHub).

In [ ]:
import glob, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
RUN = "nli_mdeberta-ft_v3_lr1e-05_e6_seed42"
found = []
for depth in range(0, 5):
    found += glob.glob("/content/drive/MyDrive/" + "*/" * depth + "model.safetensors")
good = [p for p in found if RUN in p]
if not good:
    print("❌ dossier", RUN, "introuvable. model.safetensors trouvés :", found or "aucun")
else:
    src = Path(good[0]).parent
    dst = Path("models/checkpoints") / RUN
    shutil.copytree(src, dst, dirs_exist_ok=True)
    for f in sorted(dst.iterdir()):
        print(f"{f.name:<28}{f.stat().st_size / 1e6:>10.1f} Mo")
    print("✅ checkpoint copié depuis", src)

## 5. Configuration (le moteur NLI est choisi par le script : Bronze puis affiné)

In [ ]:
os.environ.update({
    "LLM_PROVIDER": "hf",
    "LLM_MODEL": "CohereLabs/aya-expanse-8b",
    "RETRIEVAL_METHOD": "hybrid",
})
print({k: os.environ[k] for k in ("LLM_PROVIDER", "LLM_MODEL", "RETRIEVAL_METHOD")})

## 6. Essai rapide (3 affirmations du dev)
Vérifier que les **deux** NLI se chargent (sinon le script s'arrête) et que les sorties sont valides.

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict_nli --limit 3

## 7. Dev complet (20 affirmations)
**Envoyer ce résultat à Claude avant le test.**

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict_nli --splits dev

## 8. Test gelé — une seule fois

In [ ]:
!python -X utf8 -m rag_llm.silver_verdict_nli --splits test

## 9. Récupérer les résultats
Le zip contient des extraits Tunifact : **privé**, à garder sur ton PC (jamais sur GitHub).

In [ ]:
!cd evaluation/results && zip -q ../../lot-d_silver_verdict_nli.zip lot-d_silver_verdict_nli_*.json lot-d_llm_cache.jsonl
from google.colab import files
files.download("lot-d_silver_verdict_nli.zip")